This code builds on the CAR_data_import code given by Krish

In [107]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('total_call_data.csv')

C:\Users\Owner\AppData\Local\Temp\ipykernel_14332\31442611.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('total_call_data.csv')


In [3]:
# Example target activity names
target_activities = ["PreQueueMessage1", "PreQueueMessage2", "QueueMenu1", "PlayMOH300s"]

# Assuming your DataFrame is named df and has at least these columns:
# "Call ID" (or whatever uniquely identifies a call) and "Activity Name"

# Step 1: Find all call IDs that hit one of the target activities
call_ids_to_keep = df.loc[df["Activity Name"].isin(target_activities), "Contact Session ID"].unique()

# Step 2: Filter to keep *all* rows of those calls
filtered_df = df[df["Contact Session ID"].isin(call_ids_to_keep)].copy().reset_index(drop = True)

In [4]:
# Uses Queue Name
# create queue_name_df similar to menus_df but using 'Queue Name' instead of 'Activity Name'
results = []


for contact_id in filtered_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    for i in range(1, len(temp)):
        current_row = temp.iloc[i]
        prev_row = temp.iloc[i - 1]
        queue_name = current_row['Queue Name']
        if queue_name not in [np.nan, '']:
            
            results.append({'Contact Session ID': contact_id, 'Queue Name': queue_name})
            break
queue_name_df = pd.DataFrame(results)   

In [5]:
# convert Activity start Timestamp to datetime
filtered_df['Activity Start Timestamp'] = pd.to_datetime(filtered_df['Activity Start Timestamp'])

In [ ]:
filtered_df['Queue Name Identifier'] = np.nan

In [103]:
results = []

for contact_id, temp in filtered_df.groupby('Contact Session ID', sort=False):
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    name = None
    
    queue_name = temp['Queue Name'].dropna().iloc[0] if temp['Queue Name'].notna().any() else None

    single_val = temp.loc[0, 'Activity Start Timestamp']  # scalar, Timestamp object
    
    # Build a new datetime using just year, month, day
    new_date = pd.Timestamp(year=single_val.year,
                            month=single_val.month,
                            day=single_val.day)
    hour = temp.iloc[0]['hour']

    # check if any relevant activities exist
    if 'CourtesyCallback' in temp['Flow Name'].values:
        name = 'Requested Callback'
    elif 'LegalServerScreenPop' in temp['Activity Name'].values:
        name = 'Got a Call'
    else:
        name = 'No call back'
        
    # compute time in queue
    time_in_queue = 0
    prequeue_indices = temp.index[temp['Activity Name'].str.contains('PreQueue', case=False, na=False)].tolist()
    end_queue_indices = temp.index[temp['Activity Name'].str.contains('PlayMOH300s|QueueMenu1', case=False, na=False)].tolist()
    
    if prequeue_indices and end_queue_indices:
        time_in_queue = (
            temp.loc[end_queue_indices[-1], 'Activity Start Timestamp'] 
            - temp.loc[prequeue_indices[0], 'Activity Start Timestamp']
        ).total_seconds()
    
    moh_count = temp['Activity Name'].str.contains('PlayMOH300s', case=False, na=False).sum()
    menu_count = temp['Activity Name'].str.contains('QueueMenu1', case=False, na=False).sum()
    total_count = moh_count + menu_count
    
    results.append({
        'Contact Session ID': contact_id,
        'Queue Identifier': name,
        'Time in Queue (m)': time_in_queue / 60, # make minutes
        'Time Count': total_count,
        'Queue Name': queue_name,
        'Date': new_date,
        'Hour': hour
    })

final_df = pd.DataFrame(results)

for value in final_df['Queue Name'].unique():
    if final_df['Queue Name'].value_counts()[value] < 150:
        final_df.loc[final_df['Queue Name'] == value, 'Queue Name'] = 'Other'



In [106]:
final_df.to_excel('Queue_abandonment.xlsx')

## Code Below is Extra and not related to the main analysis

In [103]:
# For only large queue sizes find the time spent in each queue
large_queue_df = queue_name_df[queue_name_df['Queue Name'].isin(
    queue_name_df['Queue Name'].value_counts()[lambda x: x > 500].index
)]

# go through the filtered df and find the time spent in each queue for each contact session id
time_results = []

for contact_id in large_queue_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    for i in range(1, len(temp)):
        current_row = temp.iloc[i]
        prev_row = temp.iloc[i - 1]
        queue_name = current_row['Queue Name']
        if queue_name in large_queue_df['Queue Name'].values:
            # Calculate occurences of PlayMOH300s and QueueMenu1 in queue
            # Start if PreQueue in Activity Name
            # Count occurences of PlayMOH300s and QueueMenu1 in Activity Name
            # In order to find the time in queue, we take the time from the occurence of "PreQueue" that occurs after if we find the queue name and take it until the last occurence of "PlayMOH300s" or "QueueMenu1"
            time_in_queue = 0
            prequeue_indices = temp.index[temp['Activity Name'].str.contains('PreQueue', case=False, na=False)].tolist()
            end_queue_indices = temp.index[temp['Activity Name'].str.contains('PlayMOH300s|QueueMenu1', case=False, na=False)].tolist()
            if prequeue_indices and end_queue_indices:
                
                time_in_queue = temp.loc[end_queue_indices[-1], 'Activity Start Timestamp'] - temp.loc[prequeue_indices[0], 'Activity Start Timestamp']
                time_in_queue = time_in_queue.total_seconds()

            moh_count = temp['Activity Name'].str.contains('PlayMOH300s', case=False, na=False).sum()
            menu_count = temp['Activity Name'].str.contains('QueueMenu1', case=False, na=False).sum()
            total_count = moh_count + menu_count
            time_results.append({'Contact Session ID': contact_id, 'Queue Name': queue_name, 'Time in Queue (s)': time_in_queue, 'Time Count': total_count})
            break


In [106]:
time_results_df = pd.DataFrame(time_results)
time_results_df.head()

,Contact Session ID,Queue Name,Time in Queue (s),Time Count
0,02f3875b-ceae-4836-9786-1daf78f2841b,Consumer,376.0,2
1,06d5e9b7-9b3d-4a8a-9d50-7dfd63acaf45,SubSenior Other,891.0,6
2,093223fc-33fe-48f0-aa19-f65535bfaec8,SubSenior Other,377.0,2
3,111cf36c-81b0-4c05-9915-2cf503ad064c,Family,1020.0,6
4,11d0533f-93f8-4c56-bd15-e6100b62eb24,Consumer,698.0,4


In [143]:
time_results_df['Over 740s'] = time_results_df['Time in Queue (s)'] > 740
time_results_df['Over 1000s'] = time_results_df['Time in Queue (s)'] > 1000
time_results_df['Over 1500s'] = time_results_df['Time in Queue (s)'] > 1500

time_results_df.to_excel('queue_time_analysis.xlsx', index=False)

In [110]:
time_results_df.to_excel('queue_time_analysis.xlsx', index=False)

In [113]:
# filtering out massive outliers(more than hr wait)
filtered_time_results_df = time_results_df[time_results_df['Time in Queue (s)'] <= 3600]
filtered_time_results_df.shape
filtered_time_results_df.to_excel('filtered_queue_time_analysis.xlsx', index=False)

In [117]:
# For only medium queue sizes find the time spent in each queue
medium_queue_df = queue_name_df[(queue_name_df['Queue Name'].isin(
    queue_name_df['Queue Name'].value_counts()[lambda x: x > 200].index
)) & (queue_name_df['Queue Name'].isin(
    queue_name_df['Queue Name'].value_counts()[lambda x: x <= 500].index
))]

# go through the filtered df and find the time spent in each queue for each contact session id
time_results_med = []

for contact_id in medium_queue_df['Contact Session ID'].unique():
    temp = filtered_df.loc[filtered_df['Contact Session ID'] == contact_id].copy()
    temp = temp.sort_values('Activity Start Timestamp').reset_index(drop=True)
    
    for i in range(1, len(temp)):
        current_row = temp.iloc[i]
        prev_row = temp.iloc[i - 1]
        queue_name = current_row['Queue Name']
        if queue_name in medium_queue_df['Queue Name'].values:
            # Calculate occurences of PlayMOH300s and QueueMenu1 in queue
            # Start if PreQueue in Activity Name
            # Count occurences of PlayMOH300s and QueueMenu1 in Activity Name
            # In order to find the time in queue, we take the time from the occurence of "PreQueue" that occurs after if we find the queue name and take it until the last occurence of "PlayMOH300s" or "QueueMenu1"
            time_in_queue = 0
            prequeue_indices = temp.index[temp['Activity Name'].str.contains('PreQueue', case=False, na=False)].tolist()
            end_queue_indices = temp.index[temp['Activity Name'].str.contains('PlayMOH300s|QueueMenu1', case=False, na=False)].tolist()
            if prequeue_indices and end_queue_indices:
                
                time_in_queue = temp.loc[end_queue_indices[-1], 'Activity Start Timestamp'] - temp.loc[prequeue_indices[0], 'Activity Start Timestamp']
                time_in_queue = time_in_queue.total_seconds()

            moh_count = temp['Activity Name'].str.contains('PlayMOH300s', case=False, na=False).sum()
            menu_count = temp['Activity Name'].str.contains('QueueMenu1', case=False, na=False).sum()
            total_count = moh_count + menu_count
            time_results_med.append({'Contact Session ID': contact_id, 'Queue Name': queue_name, 'Time in Queue (s)': time_in_queue, 'Time Count': total_count})
            break

In [118]:
time_results_med_df = pd.DataFrame(time_results_med)
time_results_med_df.head()

,Contact Session ID,Queue Name,Time in Queue (s),Time Count
0,07f88374-f76b-4b0a-819e-0de67976e69a,SubSenior Homeowner,0.0,0
1,0bfab117-b496-45c8-9ead-31a176aa6a95,Family SP,721.0,4
2,0d11f1cc-fcdb-44c1-9836-d8d81f5daafe,SubSenior Homeowner,0.0,0
3,0ecd6ef4-8757-4dbd-a470-4c6ada9a394c,SubSenior Consumer,701.0,4
4,15f08be5-2c01-47ea-a088-c30887fcefb6,SubSenior Homeowner,0.0,0


In [122]:
time_results_med_df.to_excel('queue_time_analysis_med.xlsx', index=False)

In [124]:
# filtering out massive outliers(more than hr wait)
filtered_time_results_med_df = time_results_med_df[time_results_med_df['Time in Queue (s)'] <= 3600]
filtered_time_results_med_df.shape
filtered_time_results_med_df.to_excel('filtered_queue_time_analysis_med.xlsx', index=False)

In [134]:
# median of each value

for queue in time_results_df['Queue Name'].unique():
    print(queue, time_results_df.loc[time_results_df['Queue Name'] == queue, 'Time in Queue (s)'].median())


Consumer 370.0
SubSenior Other 370.0
Family 371.0
Benefits 370.0
SubSenior Tenant 370.0
Housing 370.0
SubSenior Benefits 370.0


In [138]:
filtered_time_results_df['Queue Name'].value_counts()

Queue Name
Family                1774
Consumer              1338
SubSenior Other       1152
Housing                951
Benefits               732
SubSenior Benefits     657
SubSenior Tenant       646
Name: count, dtype: int64